In [7]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, save_ca_mask)

from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)

from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Windows


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = r'F:\bmi\andres\Cohort 7\DON-014521\day0\calibration\Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-010798/day0/calibration/Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)

#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations
    
print ("DONE...")

memmap :  (72150, 512, 512)
DONE...


In [24]:
##################################################################
############### MOTION CORRECTION STEP ###########################
##################################################################
# 
print ("Skipping template makgin step: ")
#bmi_c.template = np.mean(bmi_c.data[::100],axis=0)
bmi_c.template = np.mean(bmi_c.data[:1000],axis=0)
    
print ("DONE...")

Skipping template makgin step: 
DONE...


In [25]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# convert float map to binary map using GUI
vmax = 4500
smin, smax = get_binary_std_map(bmi_c.template,vmax)
print ("...DONE...")


...DONE...


In [26]:
##################################################
############# MAKE IMAGE STD #####################
##################################################
bmi_c, img_std = get_img_std(smin, smax, bmi_c.template, bmi_c)
print ("DONE...")

max proj values (vmin, vmax):  492.1875 843.75
DONE...


In [27]:
#########################################################
########### GENERATE CELL SEGMENTATION ##################
#########################################################
min_size_roi = 100  #<---- increase to exclude really small cells
max_size_roi = 800  #<----- decrease to exclude multi-cell artificats
bmi_c.roi_centres, bmi_c.footprints = get_rois_stardist2d(img_std,
                                               min_size_roi,
                                               max_size_roi,
                                               fname)



There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
1/1 [==============================] - 0s 241ms/step


100%|███████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 77.28it/s]

idx:  34
saving:  F:\bmi\andres\Cohort 7\DON-014521\day0/day0_ca_mask.npz
